# Lesson 7.3: Rank-Based Tests: When Averages Mislead

Show how rank-based tests (Mann-Whitney, Wilcoxon, Kruskal-Wallis) cope with an outlier that a t-test cannot, using the lesson's app-timing, pain and teaching data.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Two groups: the t-test versus ranks

Task times in seconds for a new and an old app design. The old group has one 95-second outlier. Welch's t-test uses the size of every value, so the outlier hides the pattern. Mann-Whitney only uses the order of the values.

In [1]:
import numpy as np
from scipy import stats

new = [12, 15, 11, 14, 19, 13]
old = [18, 22, 25, 17, 30, 95]

t = stats.ttest_ind(new, old, equal_var=False)
mw = stats.mannwhitneyu(new, old, alternative='two-sided', method='exact')

print(f'Welch t-test : t = {t.statistic:.2f}, p = {t.pvalue:.3f}')
print(f'Mann-Whitney : U = {mw.statistic:.0f}, p = {mw.pvalue:.4f}')

Welch t-test : t = -1.67, p = 0.156
Mann-Whitney : U = 2, p = 0.0087


## 2. How strong is the Mann-Whitney effect?

U counts in how many of the 36 new-versus-old pairs the new design had the larger (slower) time. Here that is only 2, so almost every pair says the new design was faster. The rank-biserial correlation turns this into a size from 0 to 1.

In [2]:
n1, n2 = len(new), len(old)
u = mw.statistic

r = 1 - 2 * u / (n1 * n2)

print(f'new time larger in {u:.0f} of {n1 * n2} pairs')
print(f'old time larger in {n1 * n2 - u:.0f} of {n1 * n2} pairs')
print(f'rank-biserial r = {r:.2f}')

new time larger in 2 of 36 pairs
old time larger in 34 of 36 pairs
rank-biserial r = 0.89


## 3. Paired data: Wilcoxon signed-rank

Pain scores (0 to 10) for eight patients before and after treatment. The same people are measured twice, so we test the differences. The Wilcoxon statistic is the smaller of the two rank sums (positive and negative differences).

In [3]:
before = [7, 8, 6, 9, 7, 6, 8, 7]
after = [3, 1, 4, 0, 4, 7, 2, 2]

w = stats.wilcoxon(before, after)
pt = stats.ttest_rel(before, after)

print(f'Wilcoxon : W = {w.statistic:.0f}, p = {w.pvalue:.4f}')
print(f'Paired t : t = {pt.statistic:.2f}, p = {pt.pvalue:.4f}')

Wilcoxon : W = 1, p = 0.0156
Paired t : t = 3.97, p = 0.0054


## 4. Three groups: Kruskal-Wallis

Three teaching methods with three students each. Kruskal-Wallis is the rank version of one-way ANOVA. With groups this small the p-value is an approximation, so treat it as a guide.

In [4]:
a = [65, 72, 68]
b = [80, 85, 78]
c = [55, 60, 58]

kw = stats.kruskal(a, b, c)
f = stats.f_oneway(a, b, c)

print(f'Kruskal-Wallis: H = {kw.statistic:.2f}, p = {kw.pvalue:.4f}')
print(f'ANOVA         : F = {f.statistic:.2f}, p = {f.pvalue:.4f}')

Kruskal-Wallis: H = 7.20, p = 0.0273
ANOVA         : F = 38.78, p = 0.0004


## Try this

- Change the 95 to 40 and then to 5000. Predict what happens to the Welch p-value and to the Mann-Whitney p-value, then check.
- In the pain data, change patient 6's After score from 7 to 5. Does the Wilcoxon statistic get smaller or larger?

## The same idea in R, SPSS and Stata

**R**

```r
new <- c(12, 15, 11, 14, 19, 13)
old <- c(18, 22, 25, 17, 30, 95)
t.test(new, old)
wilcox.test(new, old)
before <- c(7, 8, 6, 9, 7, 6, 8, 7)
after <- c(3, 1, 4, 0, 4, 7, 2, 2)
wilcox.test(before, after, paired = TRUE)
score <- c(65, 72, 68, 80, 85, 78, 55, 60, 58)
method <- factor(rep(c("A", "B", "C"), each = 3))
kruskal.test(score ~ method)
```

**SPSS** (Analyze > Nonparametric Tests > Legacy Dialogs > 2 Independent Samples (Mann-Whitney U) and 2 Related Samples (Wilcoxon))

```
DATA LIST FREE / time group.
BEGIN DATA
12 1  15 1  11 1  14 1  19 1  13 1  18 2  22 2  25 2  17 2  30 2  95 2
END DATA.
NPAR TESTS /M-W=time BY group(1,2).
DATA LIST FREE / before after.
BEGIN DATA
7 3  8 1  6 4  9 0  7 4  6 7  8 2  7 2
END DATA.
NPAR TESTS /WILCOXON=before WITH after (PAIRED).
```

**Stata**

```
clear
set obs 12
generate time = real(word("12 15 11 14 19 13 18 22 25 17 30 95", _n))
generate group = _n > 6
ranksum time, by(group)
ttest time, by(group) unequal
generate before = real(word("7 8 6 9 7 6 8 7", _n))
generate after = real(word("3 1 4 0 4 7 2 2", _n))
signrank before = after
generate score = real(word("65 72 68 80 85 78 55 60 58", _n))
generate method = ceil(_n / 3) if _n <= 9
kwallis score, by(method)
```